# 编译配置 — 融合、精度、buffer、stream

上一节我们把图准备好了。本节进入"怎么编译这张图"——同一张图，配置不同，编译出的 OM 在**性能**和**精度**上可能差异很大。GE 把编译配置收敛为一套 key-value 选项体系，本节带你掌握其中最关键的四类：**融合、精度、buffer/内存、stream**，并讲清这些选项从哪传入、谁覆盖谁。

本节学习大纲如下：

- 配置体系总览（初始化选项 / 编译选项 / Session·图级选项）
- 融合配置（开关文件、一键关闭、按 Pass 控制）
- 精度配置（PRECISION_MODE / PRECISION_MODE_V2）
- Buffer 与内存配置（数据缓存优化、内存复用）
- Stream 与引擎配置
- 综合配置示例与取舍

> 重要前提：本节面向**用户怎么配**，不深入 GE 内部融合/stream 分配的实现。所有选项名均取自 GE 真实文档（`aclgrphBuildInitialize` / `aclgrphBuildModel` 支持的配置参数、options 参数说明）。

## 1. 配置体系总览

GE 的编译配置通过 **map<key, value>** 传入，分布在三个层面——**初始化选项**（global_options）、**编译选项**（build options）、**Session / 图级选项**（GeSession options）。离线编译主要用前两者；在线编译（GeSession）用第三者。

### 1.1 三类选项与传入入口

<p align="center"><img src="./images/compile_config_overview.svg" alt="三类编译选项与传入入口" width="60%"></p>

| 入口 | key 风格 | 配置文档 |
| --- | --- | --- |
| `aclgrphBuildInitialize(global_options)` | `ge::ir_option::XXX` | aclgrphBuildInitialize 支持的配置参数 |
| `aclgrphBuildModel(graph, build_options, model)` | `ge::ir_option::XXX` | aclgrphBuildModel 支持的配置参数 |
| `GeSession(...)` / `AddGraph(..., graph_options)` | 文档规定的原始字符串（多数为 `"ge.*"`，也有例外） | options 参数说明 |

> `ge::ir_option::XXX` 是 C++ 常量名；Python 需要传入该常量对应的原始字符串，不能自行删改前缀。例如 `PRECISION_MODE_V2` 对应 `"ge.exec.precision_mode_v2"`，而 `AC_PARALLEL_ENABLE` 对应 `"ac_parallel_enable"`，并非所有 key 都以 `ge.` 开头。

### 1.2 哪些放初始化、哪些放编译

- **放初始化（global_options）**：影响整个编译进程的、一次性的，如 `SOC_VERSION`（无卡编译必填）、`BUFFER_OPTIMIZE`（数据缓存优化开关）、`FUSION_SWITCH_FILE`（融合开关文件）。
- **放编译（build_options）**：针对单张图的，如 `INPUT_SHAPE`、`INPUT_FORMAT`、`OUTPUT_TYPE`、`PRECISION_MODE_V2`、`OPTIMIZATION_SWITCH`、`EXTERNAL_WEIGHT`。

> 优先级直觉：**越靠近"这张图"的配置，优先级越高**。在线场景 options 可声明生效范围为 global / session / graph（见 options 参数说明的"全局/session/graph 级别生效"列），graph 级覆盖 session 级、session 级覆盖 global 级。注意 `PRECISION_MODE` / `PRECISION_MODE_V2` 在多图场景下各图取值需保持一致。

## 2. 融合配置

算子融合（Fusion）是 GE 把多个小算子合并成大算子、减少访存与调度开销的关键优化，**默认开启**。绝大多数场景不用动它；只有当某条融合规则导致功能/精度异常、需要定位时，才会去关闭它。GE 提供三种粒度的融合控制。

### 2.1 融合开关文件（FUSION_SWITCH_FILE）

在 `aclgrphBuildInitialize` 里传入一个 JSON 开关文件，按 Pass 名 `on`/`off` 控制图融合（GraphFusion）和 UB 融合（UBFusion）：

```cpp
{ge::ir_option::FUSION_SWITCH_FILE, "/home/test/fusion_switch.cfg"}
```

`fusion_switch.cfg` 内容示例（关闭某条规则）：

```json
{
  "Switch": {
    "GraphFusion": {
      "ConvToFullyConnectionFusionPass": "off",
      "SoftmaxFusionPass": "on"
    },
    "UBFusion": {
      "TbePool2dQuantFusionPass": "on"
    }
  }
}
```

### 2.2 一键关闭融合（ALL）

定位问题时常用"先全关、再逐个打开"的二分法：

```json
{
  "Switch": {
    "GraphFusion": { "ALL": "off", "SoftmaxFusionPass": "on" },
    "UBFusion":    { "ALL": "off", "TbePool2dQuantFusionPass": "on" }
  }
}
```

> 注意：`ALL:off` 是"**部分**一键关闭"——出于系统机制，部分融合规则无法关闭；关闭某些规则可能引发功能问题。可关闭的规则清单见《图融合和 UB 融合规则参考》。

### 2.3 按 Pass 精确控制（OPTIMIZATION_SWITCH）

在 `aclgrphBuildModel` 编译选项里，用 `OPTIMIZATION_SWITCH` 对单个或多个融合 Pass 直接开关：

```cpp
{ge::ir_option::OPTIMIZATION_SWITCH, "Passname1:on;Passname2:off"}
```

多组用分号分隔，key 为 Pass 名、value 为 on/off（不支持大小写模糊匹配）。

### 2.4 验证融合是否生效

`OPTION_EXPORT_COMPILE_STAT` 控制是否生成融合结果文件 `fusion_result.json`：

- `0`：不生成。
- `1`（默认值）：程序正常退出时生成。
- `2`：图编译完成时生成；即使后续程序提前中断，已完成编译的图仍有结果文件。

文件会记录每条规则的 `match_times`（匹配次数）和 `effect_times`（实际生效次数），用于核对某条融合到底有没有命中。

> 对 TensorFlow 模型，Parser 阶段还有 Scope 融合，用解析参数 `ENABLE_SCOPE_FUSION_PASSES` 指定生效的 Non-General Scope 规则（见上一节）。

## 3. 精度配置

精度配置控制"算子用什么数据类型算"，是**性能与精度的权衡旋钮**。GE 提供两套选项：旧版 `PRECISION_MODE` 与新版 `PRECISION_MODE_V2`（**推荐用 V2**，语义更清晰）。同一张图中二者**不能同时使用**。

### 3.1 PRECISION_MODE_V2 取值（推荐）

```cpp
{ge::ir_option::PRECISION_MODE_V2, "fp16"}
```

| 取值 | 含义 | 取向 |
| --- | --- | --- |
| `fp16`（默认值） | 原图 fp16/bf16/fp32 算子强制选 float16 | 性能优先 |
| `origin` | 保持原图精度 | 精度优先 |
| `cube_fp16in_fp32out` | cube（矩阵）类算子优先 fp16 输入 / fp32 输出 | 精度+性能折中 |
| `mixed_float16` | 混合精度：按内置策略把部分 fp32/bf16 算子降到 fp16 | 性能优先（兼顾精度） |
| `mixed_bfloat16` | 混合精度：按内置策略把部分 fp32 算子降到 bf16 | 性能优先（兼顾精度、降低内存） |
| `mixed_hif8` | 混合精度：按内置策略把部分 fp16/bf16/fp32 算子降到 hifloat8 | 性能优先（兼顾精度、降低内存） |
| `cube_hif8` | cube 算子同时支持多种精度时强制选择 hifloat8 | 性能优先 |

> 产品限制：bf16 仅支持 Atlas A2 训练/推理系列、Atlas A3 训练/推理系列、Atlas 200I/500 A2 推理产品以及 Ascend 950PR/950DT；hif8 仅支持 Ascend 950PR/950DT。本教程目前只在 Atlas A2 系列验证，因此 A2 环境不要配置 `mixed_hif8` 或 `cube_hif8`。

> 默认 `fp16` 为性能优先，推理可能出现精度溢出；**若遇到精度问题，先把它改成 `origin` 或某种混合精度**复测。

### 3.2 PRECISION_MODE 取值（旧版，了解即可）

```cpp
{ge::ir_option::PRECISION_MODE, "force_fp16"}
```

常见取值：`force_fp16`、`must_keep_origin_dtype`（保持原精度）、`allow_fp32_to_fp16`、`allow_mix_precision` / `allow_mix_precision_fp16`、`allow_mix_precision_bf16`、`allow_fp32_to_bf16`、`force_fp32`。

### 3.3 混合精度的黑白灰名单

开启 `mixed_*` / `allow_mix_*` 后，每个算子是否允许降精度由内置策略文件里的 `precision_reduce` 决定：

| 名单 | precision_reduce | 行为 |
| --- | --- | --- |
| 白名单 | `true` | 允许降精度（如 fp32→fp16） |
| 黑名单 | `false` | 不允许降精度，保持原精度 |
| 灰名单 | 未配置 | 跟随前一个算子的降精度决策 |

可用 `MODIFY_MIXLIST` 指定自定义黑白灰名单 JSON，精细决定哪些算子降、哪些不降。

### 3.4 按算子设精度（OP_PRECISION_MODE）

需要对个别算子单独调精度时，用 `OP_PRECISION_MODE` 传入 `op_precision.ini`，可按算子类型或节点名设 `high_precision` / `high_performance`：

```ini
[ByOpType]
MatMul=high_precision

[ByNodeName]
matmul_1=high_precision
```

`[ByNodeName]` 用于把解析模式切换到节点名；如果省略，节点名会被当成算子类型而静默不生效。示例中的 `MatMul` 和 `matmul_1` 应替换为图里的真实算子类型和节点名。

> 精度调优总思路：**先用默认（性能优先）跑通；出现精度问题时，逐级提精度**——整图 `PRECISION_MODE_V2=origin` → 混合精度 + 黑白名单 → 个别算子 `OP_PRECISION_MODE=high_precision`，在满足精度的前提下尽量保留性能。

## 4. Buffer 与内存配置

这一类配置影响 OM 运行时的**内存占用与访存效率**。分两个层面：**数据缓存优化（buffer）** 和 **内存复用/分配策略**。

### 4.1 数据缓存优化（BUFFER_OPTIMIZE）

在 `aclgrphBuildInitialize` 里设置，通过高速缓存暂存数据提升性能，**默认开启 l2 优化**：

```cpp
{ge::ir_option::BUFFER_OPTIMIZE, "l2_optimize"}
```

| 取值 | 含义 |
| --- | --- |
| `l2_optimize`（默认） | 开启 L2 数据缓存优化 |
| `off_optimize` | 关闭数据缓存优化 |
| `l1_optimize` | 当前版本无效，等同 `off_optimize` |

> 建议保持开启。个别算子在某些场景下的缓存优化可能影响精度——**出现精度问题时可尝试关闭**（`off_optimize`）；若关闭后精度达标，需识别问题算子反馈技术支持，解决后再开回。

### 4.2 内存复用（EXEC_DISABLE_REUSED_MEMORY / ge.exec.disableReuseMemory）

内存复用：按生命周期和大小，把不冲突的内存重复使用，降低网络内存占用，**默认开启**。

```cpp
{ge::ir_option::EXEC_DISABLE_REUSED_MEMORY, "0"}   // 0=开启复用(默认), 1=关闭复用
```

在线 options 对应 `ge.exec.disableReuseMemory`（同样 0 开 1 关）。

> 警告：关闭内存复用（设 1）会导致 Device 侧内存不复用，**大模型可能因此内存不足**。一般不要关。

### 4.3 内存分配策略与峰值优化（在线 options）

GeSession 在线场景还有更细的内存选项：

| 选项 | 作用 |
| --- | --- |
| `ge.exec.staticMemoryPolicy` | 0=动态分配且不扩展（默认）；1=兼容值，按 2 处理；2=静态 shape 内存动态扩展；3=仅动态 shape 内存动态扩展；4=静态、动态 shape 均支持内存动态扩展 |
| `ge.exec.inputReuseMemIndexes` | 把指定输入节点的内存复用为中间内存，降低内存峰值（输入需 32 字节对齐） |
| `ge.exec.outputReuseMemIndexes` | 把整图输出内存复用为中间内存，降低内存峰值 |
| `ge.exec.atomicCleanPolicy` | memset 内存集中清理(0,默认) / 单独清理(1)，大 memset 时降内存 |

> `staticMemoryPolicy=2/4` 可让同一 Session 中的多张图按最大需求复用静态 shape 内存，但不支持多图并发执行；`3/4` 用于减少动态 shape 内存碎片并降低占用，可能带来一定性能损失。

> 使用建议：**buffer 优化和内存复用默认就是好配置**，平时不用动；只在"显存不够"或"怀疑缓存优化影响精度"时，才有针对性地调整这几个开关。

## 5. Stream 与引擎配置

Stream（流）是任务下发与并行执行的载体。GE 在图编译期会自动完成 stream 分配，**用户一般无需手工指定 stream 数量**；可调的是"是否让不同引擎的算子并行""哪些引擎参与编译""是否开启多流并行优化"。

### 5.1 多引擎并行（AC_PARALLEL_ENABLE）

动态 shape 图中，开启后系统自动识别可与 AI Core 并发的 AI CPU 算子，把它们下发到不同的流上，实现引擎间并行、提升动态 shape 执行性能：

```cpp
{ge::ir_option::AC_PARALLEL_ENABLE, "1"}   // 1=允许 AI CPU 与 AI Core 并行; 0=默认不单独分流
```

### 5.2 排除指定引擎（EXCLUDE_ENGINES）

NPU 集成多种加速引擎（AiCore / AiVec / AiCpu，按优先级排列）。图编译时会为算子自动选优先级最高的引擎。`EXCLUDE_ENGINES` 可排除某些引擎，多个用 `|` 分隔：

```cpp
{ge::ir_option::EXCLUDE_ENGINES, "AiCore|AiVec"}
```

| 引擎取值 | 含义 |
| --- | --- |
| `AiCore` | AI Core 硬件加速引擎 |
| `AiVec` | Vector Core 硬件加速引擎 |
| `AiCpu` | AI CPU 硬件加速引擎 |

典型用法：训练中给数据预处理图配置不使用 AiCore，避免和主训练图抢占 AI Core。

### 5.3 多流并行模式（ge.autoMultistreamParallelMode）

> 该参数为调试功能扩展参数，当前不支持应用于商用产品中，后续版本会作为正式功能更新发布。

该参数仅适用于静态 shape 图场景，开发者可通过配置此参数控制多流并行模式的自动分配策略，以提升图执行性能。生效级别为 session/graph。

| 取值 | 含义 |
| --- | --- |
| `None`（默认值） | 不启用任何多流并行优化 |
| `cv` | 开启 Cube 算子与 Vector 算子的并行执行 |

```cpp
{"ge.autoMultistreamParallelMode", "cv"}
```

> 注意：该参数仅限推荐类型网络的训练场景使用；不能与多流并发执行功能（`ENABLE_DYNAMIC_SHAPE_MULTI_STREAM`）同时使用。

> 关于 stream 与多引擎的内部分配机制（流分配、内存冲突避免等）属于 GE 内部实现，用户侧只需理解：**这些选项是"让谁参与、是否并行"的开关**，stream 的具体编排交给 GE 自动完成。

## 6. 综合配置示例与取舍

把前面几类选项放到一起，看一个贴近实战的离线编译配置（C++ Build 接口）：

```cpp
// 初始化级选项（aclgrphBuildInitialize）
std::map<ge::AscendString, ge::AscendString> global_options = {
    {ge::ir_option::SOC_VERSION,     "Ascend910B1"},   // 无卡编译必填
    {ge::ir_option::BUFFER_OPTIMIZE, "l2_optimize"},   // 数据缓存优化（默认开）
};
ge::aclgrphBuildInitialize(global_options);

// 编译级选项（aclgrphBuildModel）
std::map<ge::AscendString, ge::AscendString> build_options = {
    {ge::ir_option::INPUT_FORMAT,        "ND"},
    {ge::ir_option::INPUT_SHAPE,         "data:2,3"},
    {ge::ir_option::PRECISION_MODE_V2,   "fp16"},       // 性能优先；精度异常改 origin
    {ge::ir_option::OUTPUT_TYPE,         "FP32"},
    {ge::ir_option::OPTIMIZATION_SWITCH, "Passname2:off"}, // 定位时关掉某条融合
};
ge::ModelBufferData model;
ge::aclgrphBuildModel(graph, build_options, model);
```

Python 侧（第一章离线样例风格）同理，`build_options` 是一个字典：

```python
from ge.offline_compile import build_model

# 假设前文已经完成 build_initialize，且 graph 已构建。
build_options = {
    "input_format": "ND",
    "ge.exec.precision_mode_v2": "fp16",
}
model = build_model(graph, build_options)
```

### 配置取舍速查

| 目标 | 推荐配置 |
| --- | --- |
| 默认追求性能 | 不改：融合开、`PRECISION_MODE_V2=fp16`、buffer/内存复用默认开 |
| 出现精度问题 | `PRECISION_MODE_V2=origin` 或 `mixed_*` + 黑白名单；可试关 `BUFFER_OPTIMIZE` |
| 显存不足 | 保持内存复用开启；在线可调 `ge.exec.staticMemoryPolicy` / `inputReuseMemIndexes` |
| 定位某条融合问题 | `FUSION_SWITCH_FILE` 关规则，或 `OPTIMIZATION_SWITCH` 按 Pass 关 |
| OM 体积受限 | `EXTERNAL_WEIGHT=1` 外置权重（详见 3.4 节） |

> 使用建议：**默认配置已经是经过调优的"好起点"**。先用默认编译跑通、看性能/精度，再"按问题对症下药"地改单个选项，而不是一上来就堆一堆配置。

## 7. 动手实践：让真实编译器接收配置并执行

下面不再只生成配置字典：精度和内存复用选项通过 `GeApi.ge_initialize` 传入，Buffer 优化通过 `Session.add_graph(..., graph_options)` 传给当前图，随后在线编译 Add 图并在 0 号 NPU 上执行。编译成功且结果对拍通过，可以证明图在所给配置下正确运行；优化是否带来性能或内存收益仍需通过 Profiling 或对照实验确认。

运行前需已配置 CANN 环境并可访问 0 号 NPU。示例只设置新版 `ge.exec.precision_mode_v2`，没有同时设置旧版 `ge.exec.precision_mode`，从源头避免互斥配置。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
# === 真机运行：应用编译配置 -> 在线编译 -> NPU 执行 ===
import json

import numpy as np
from ge.es.graph_builder import GraphBuilder
from ge.ge_global import GeApi
from ge.graph import Tensor
from ge.graph.types import DataType, Format
from ge.session import Session

DEVICE_ID = 0
GRAPH_ID = 1

# 这些是 GE 在线编译使用的真实 option key，按公开生效层级传入。
global_options = {
    "ge.exec.deviceId": str(DEVICE_ID),
    "ge.graphRunMode": "0",
    "ge.exec.precision_mode_v2": "origin",
    "ge.exec.disableReuseMemory": "0",
}
graph_options = {
    "ge.bufferOptimize": "l2_optimize",
}

builder = GraphBuilder("ConfiguredAddGraph")
x = builder.create_input(
    index=0, name="input_x", data_type=DataType.DT_FLOAT, shape=[2, 3]
)
y = builder.create_input(
    index=1, name="input_y", data_type=DataType.DT_FLOAT, shape=[2, 3]
)
builder.set_graph_output(x + y, 0)
graph = builder.build_and_reset()

input_x = np.arange(6, dtype=np.float32).reshape(2, 3)
input_y = np.full((2, 3), 0.5, dtype=np.float32)
expected = input_x + input_y

ge_api = GeApi()
ge_api.ge_initialize(global_options)
session = None
try:
    session = Session()
    session.add_graph(GRAPH_ID, graph, graph_options)

    inputs = [
        Tensor(input_x.reshape(-1).tolist(), None, DataType.DT_FLOAT, Format.FORMAT_ND, [2, 3]),
        Tensor(input_y.reshape(-1).tolist(), None, DataType.DT_FLOAT, Format.FORMAT_ND, [2, 3]),
    ]
    outputs = session.run_graph(GRAPH_ID, inputs)
    actual = np.asarray(outputs[0].data, dtype=np.float32)
    np.testing.assert_allclose(actual, expected, rtol=1e-6, atol=1e-6)

    print("提交给 GE 的全局配置：")
    print(json.dumps(global_options, ensure_ascii=False, indent=2))
    print("提交给当前图的配置：")
    print(json.dumps(graph_options, ensure_ascii=False, indent=2))
    print("[OK] 图在上述配置下完成编译、执行并通过数值校验")
    print("[INFO] 优化收益需要通过 Profiling 或对照实验确认")
    print(actual)
finally:
    outputs = None
    inputs = None
    # 释放 Session 引用，由 Session 析构统一释放图资源。
    session = None
    ge_api.ge_finalize()


## 课后练习

本节讲了编译配置三层体系，以及融合、精度、buffer/内存、stream 四类关键配置。请完成以下题目自测。

1. （判断题）`PRECISION_MODE` 与 `PRECISION_MODE_V2` 可以在同一张图中同时使用。

2. （判断题）内存复用（`EXEC_DISABLE_REUSED_MEMORY`）默认开启；关闭它在大模型场景可能导致 Device 内存不足。

3. （单选题）`PRECISION_MODE_V2` 的默认值及其取向是？
    A. `origin`，精度优先
    B. `fp16`，性能优先
    C. `mixed_bfloat16`，省内存优先
    D. `force_fp16`，精度优先

4. （单选题）想关闭某条具体的图融合规则进行问题定位，以下哪种方式不合适？
    A. 用 `FUSION_SWITCH_FILE` 配置文件把该 Pass 设为 off
    B. 用 `OPTIMIZATION_SWITCH` 设 `该Pass:off`
    C. 用 `FUSION_SWITCH_FILE` 的 `ALL:off` 再单独打开需要的 Pass
    D. 修改 `PRECISION_MODE_V2` 为 origin

5. （多选题）以下关于 GE 编译配置体系的描述，哪些是正确的？
    A. `SOC_VERSION` 等初始化级选项通过 `aclgrphBuildInitialize` 传入
    B. `INPUT_SHAPE`、`PRECISION_MODE_V2` 等编译级选项通过 `aclgrphBuildModel` 传入
    C. GeSession 在线 options 使用文档规定的原始字符串 key（多数为 `ge.*`，也有 `ac_parallel_enable` 等例外），并可按文档声明的 global/session/graph 层级生效
    D. 所有配置都只能在 `aclgrphBuildInitialize` 里设置

6. （多选题）以下关于 buffer 与内存配置的描述，哪些是正确的？
    A. `BUFFER_OPTIMIZE` 默认 `l2_optimize`，开启数据缓存优化
    B. 出现精度问题时，可尝试把 `BUFFER_OPTIMIZE` 设为 `off_optimize` 复测
    C. `EXEC_DISABLE_REUSED_MEMORY` 设为 1 表示开启内存复用
    D. 在线场景可用 `ge.exec.inputReuseMemIndexes` 复用输入内存以降低内存峰值

7. （单选题）关于 `EXCLUDE_ENGINES` 与 `AC_PARALLEL_ENABLE`，以下说法正确的是？
    A. `EXCLUDE_ENGINES` 用于增加 stream 数量
    B. `AC_PARALLEL_ENABLE` 开启后可让 AI CPU 与 AI Core 算子并行（动态 shape 图）
    C. 用户必须手工指定 stream 数量，GE 不会自动分配
    D. `EXCLUDE_ENGINES` 多个引擎之间用分号分隔

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.03_answer.txt